# Step 7 — Swap in embeddings

*Step 7 of the AI in Industry lab*

---

## Read this before you run anything

You will swap the word-matching retriever for one that matches on meaning, and compare what each one finds.

**What you should end up understanding:** What embeddings are for — and that swapping in a fancier technique does not automatically make a system better.

| | |
|---|---|
| **Cost** | 1 API call |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes. The first run is slower because it loads a small model. |

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b>Uses about 1 API call.</b> Re-running cells is fine, it just uses a little more of your free quota each time.</div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; Honest warning: on this corpus embeddings do NOT clearly beat the simpler method. That is a real result, not a broken notebook. It is exactly why step 11 exists.</b></div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

Step 6(c) showed TF-IDF failing on *"exemption"* vs *"condoned"*.

**Embeddings** turn text into a list of numbers where similar meanings land near each other, so a match does not need shared words. The vectors for all 227 clauses ship with the lab, already computed.

In [ ]:
from labcore import corpus, tfidf, embed, grounded

chunks, texts, _ = corpus()
by_word    = tfidf(texts)     # step 4's retriever
by_meaning = embed(texts)     # the new one

PARAPHRASE = "Can I get an exemption if I miss too many classes?"

print("BY WORD    ->", by_word(PARAPHRASE, k=1)[0][0][:90])
print()
print("BY MEANING ->", by_meaning(PARAPHRASE, k=1)[0][0][:90])

## An honest result

You may find that **embeddings do not obviously win here.** That is not a bug in this notebook — it is a real measurement.

We tested six different phrasings across two embedding models. Neither reliably beat TF-IDF on this corpus, and sometimes they did worse. The right clause *is* in there; it just ranks below other attendance clauses that look equally plausible to both methods.

**This is the honest state of the field**, and it is why the last step exists: you cannot tell which retriever is better by looking at one example. You have to measure.

## Ask with the new retriever

In [ ]:
print(grounded(by_meaning)(PARAPHRASE, k=4))

---

## Now change it yourself

Find a question where the two retrievers genuinely disagree.

In [ ]:
my_question = "What if I am sick and cannot attend for two weeks?"

print("BY WORD:")
for t, s in by_word(my_question, k=2):
    print(f"   {s:.3f}  {t[:80]}...")

print("\nBY MEANING:")
for t, s in by_meaning(my_question, k=2):
    print(f"   {s:.3f}  {t[:80]}...")

# The scores are not comparable between the two - different scales.
# What matters is WHICH chunks come back, and in what order.

---

### Done with step 7

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.